In [ ]:
import torch
import torch.nn as nn

# ====================================================
# 1. Channel Attention Module
# ====================================================
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        # 가로세로를 1x1로 찌그러뜨려서 픽셀 대신 '전체적인 채널 정보'만 봅니다.
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        # 모델의 가벼움을 위해 채널을 1/16로 확 줄였다가 다시 늘려줍니다 (핵심!)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        # 중요도를 0~1 사이의 확률값(점수)으로 만듭니다.
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        # 평균값과 최대값의 결과를 더한 뒤, 시그모이드로 점수를 매깁니다.
        attention_score = self.sigmoid(avg_out + max_out)

        # 원본 피처(x)에 중요도 점수(0~1)를 곱해서 내보냅니다! (노이즈는 0에 가깝게 곱해짐)
        return x * attention_score


# ====================================================
# 2. Spatial Attention Module
# ====================================================
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        # 픽셀 위치를 훑어볼 커다란 컨볼루션 창문을 만듭니다.
        padding = 3 if kernel_size == 7 else 1
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 겹쳐져 있는 채널들을 가로세로 '한 판'으로 눌러서(평균, 최대) 특징 위치만 뽑습니다.
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        # 평균판과 최대판을 겹친 뒤, 창문(conv)으로 훑습니다.
        concat = torch.cat([avg_out, max_out], dim=1)
        attention_score = self.sigmoid(self.conv1(concat))

        # 원본 피처(x) 위치에 중요도 점수를 곱해줍니다.
        return x * attention_score


# ====================================================
# 3. CBAM 최종 조립 팩
# ====================================================
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super(CBAM, self).__init__()
        # 두 개의 돋보기를 직렬로 조립!
        self.channel_gate = ChannelAttention(in_channels, reduction)
        self.spatial_gate = SpatialAttention(kernel_size)

    def forward(self, x):
        # 채널 먼저 거르고, 그다음 공간을 거릅니다. (논문 정석 순서)
        x_out = self.channel_gate(x)
        x_out = self.spatial_gate(x_out)
        return x_out

print("✅ CBAM 클래스 구현 완료!")

✅ CBAM 클래스 구현 완료!


In [ ]:
# CompressAI 모델 안에서는 텐서들이 대충 (1묶음, 64채널, 가로세로 64) 크기로 돌아다닙니다.
# 그걸 모방해서 임의의 랜덤(더미) 텐서를 하나 만들어줍니다.
dummy_tensor = torch.randn(1, 64, 64, 64)
print(f"👉 통과 전(Input) 텐서 크기: {dummy_tensor.shape}")

# 우리가 만든 CBAM 모듈을 장착! (채널이 64니까 64를 인자로 줍니다)
my_attention_module = CBAM(in_channels=64, reduction=16)

# 더미 텐서를 CBAM 문(모듈)에 억지로 통과시켜 봅니다.
output_tensor = my_attention_module(dummy_tensor)
print(f"👈 통과 후(Output) 텐서 크기: {output_tensor.shape}")

# CBAM의 철학: 원본 데이터의 크기를 단 1도 바꾸지 않고 '필터링'만 한다!
assert dummy_tensor.shape == output_tensor.shape
print("🎉 성공!! 입력 크기와 출력 크기가 동일합니다. 완벽한 플러그-앤-플레이 모듈입니다!")

👉 통과 전(Input) 텐서 크기: torch.Size([1, 64, 64, 64])
👈 통과 후(Output) 텐서 크기: torch.Size([1, 64, 64, 64])
🎉 성공!! 입력 크기와 출력 크기가 동일합니다. 완벽한 플러그-앤-플레이 모듈입니다!


In [ ]:
from compressai.zoo import bmshj2018_hyperprior

print("🚀 베이스 모델을 불러옵니다...")
model = bmshj2018_hyperprior(quality=2, pretrained=True)

# 1. 수술할 위치 파악 (CompressAI는 모델의 특징 추출 채널 개수를 'M'이라는 변수에 저장해둡니다)
channels = model.M  # quality=2 에서는 보통 192개의 채널이 튀어나옵니다.
print(f"🔍 병목(Bottleneck) 직전의 피처맵 채널 개수는 {channels}개 입니다.")

# 2. 이 채널 크기에 딱 맞춘 우리만의 CBAM 부품(레고)을 생성!
my_cbam = CBAM(in_channels=channels, reduction=16)

# ====================================================
# 🪚 [본격적인 절개 및 접합 수술 (단 3줄!)]
# ====================================================
# 기존 CompressAI 모델에서 이미지가 지나가는 첫 번째 길(Encoder, g_a)의 구조물들을 리스트로 쫙 빼옵니다.
encoder_layers = list(model.g_a.children())

# 그 길의 맨 끝(Bottleneck 직전)에 우리의 CBAM 문을 하나 살포시 끼워 넣습니다.
encoder_layers.append(my_cbam)

# 문을 하나 더 추가한 그 길을, 다시 모델의 Encoder(g_a)로 통째로 덮어씌웁니다! (수술 부위 봉합)
model.g_a = nn.Sequential(*encoder_layers)
# ====================================================

print("🎉 바뀐 Encoder(g_a)의 끝부분 구조를 확인해볼까요?")

# 모델 구조 출력 (가장 마지막 모듈을 확인해보세요!)
print("\n" + "="*50)
print(model.g_a)
print("="*50)

🚀 [수술실] 베이스 모델(환자)을 불러옵니다...
Downloading: "https://compressai.s3.amazonaws.com/models/v1/bmshj2018-hyperprior-2-93677231.pth.tar" to /root/.cache/torch/hub/checkpoints/bmshj2018-hyperprior-2-93677231.pth.tar


100%|██████████| 20.2M/20.2M [00:00<00:00, 22.1MB/s]


🔍 병목(Bottleneck) 직전의 피처맵 채널 개수는 192개 입니다.
🎉 이식 수술 대성공! 바뀐 Encoder(g_a)의 끝부분 구조를 확인해볼까요?

Sequential(
  (0): Conv2d(3, 128, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
  (1): GDN(
    (beta_reparam): NonNegativeParametrizer(
      (lower_bound): LowerBound()
    )
    (gamma_reparam): NonNegativeParametrizer(
      (lower_bound): LowerBound()
    )
  )
  (2): Conv2d(128, 128, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
  (3): GDN(
    (beta_reparam): NonNegativeParametrizer(
      (lower_bound): LowerBound()
    )
    (gamma_reparam): NonNegativeParametrizer(
      (lower_bound): LowerBound()
    )
  )
  (4): Conv2d(128, 128, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
  (5): GDN(
    (beta_reparam): NonNegativeParametrizer(
      (lower_bound): LowerBound()
    )
    (gamma_reparam): NonNegativeParametrizer(
      (lower_bound): LowerBound()
    )
  )
  (6): Conv2d(128, 192, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2))
  (7): CBAM(
    (channel_gate): Channel